In [31]:
import pandas as pd
import numpy as np
import string
import nltk
import re
import sklearn.utils as sk
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.naive_bayes import ComplementNB
from sklearn.metrics import classification_report

from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from nltk.tokenize import word_tokenize

In [15]:
import ssl
import nltk

try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

nltk.download('stopwords')


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/salmaameer/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [16]:
# Download NLTK data
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/salmaameer/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [17]:
# Load dataset
df = pd.read_csv('../amazon_reviews.csv')
df = sk.shuffle(df)
print(df.head())

      sentiments                                     cleaned_review
10207   negative  ok for gaming the sound is already failing on ...
4578     neutral  had it for three months and it was working fin...
12664   positive  i haven tried it yet but it looks good and fee...
3892    positive  this is so pretty fits my hand perfect the lig...
15739   positive  this is powerful little speaker it easily conn...


**1- Preprocessing Data**

In [18]:
def preprocess_text(text):
    ps = PorterStemmer()
    stop_words = set(stopwords.words('english'))
    tokens = word_tokenize(text.lower())
    filtered_tokens = [word for word in tokens if word.isalnum() and word not in stop_words] # remove stop words
    stemmed_tokens = [ps.stem(word) for word in filtered_tokens] # stemming

    return ' '.join(stemmed_tokens)


df['cleaned_review'] = df['cleaned_review'].fillna('') # fill missing values with empty string
df['cleaned_review'] = df['cleaned_review'].apply(preprocess_text)
print(df.head())


      sentiments                                     cleaned_review
10207   negative     ok game sound alreadi fail one side disappoint
4578     neutral  three month work fine love click quiet lightwe...
12664   positive  tri yet look good feel snug imagin long game s...
3892    positive  pretti fit hand perfect light soooo pretti lol...
15739   positive  power littl speaker easili connect mani devic ...


**2- Labeling**

In [19]:
def mappingToNumbers():
  categories = {"negative": 0, "neutral": 1, "positive": 2}
  df['sentiments'] = df['sentiments'].map(categories)
  print(df.head())

mappingToNumbers()

       sentiments                                     cleaned_review
10207           0     ok game sound alreadi fail one side disappoint
4578            1  three month work fine love click quiet lightwe...
12664           2  tri yet look good feel snug imagin long game s...
3892            2  pretti fit hand perfect light soooo pretti lol...
15739           2  power littl speaker easili connect mani devic ...


**3- Data splitting**

In [36]:
X_train, X_test, y_train, y_test = train_test_split(df['cleaned_review'], df['sentiments'], test_size=0.2, random_state=42)

**4- TF-IDF vectorizer**

In [37]:
vectorizer = TfidfVectorizer()
X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

**5.1: SVM Model**

In [40]:
svmModel = SVC()
svmModel.fit(X_train, y_train)
yPred = svmModel.predict(X_test)
print("SVM Classification Report:\n", classification_report(y_test, yPred))

SVM Classification Report:
               precision    recall  f1-score   support

           0       0.89      0.48      0.62       308
           1       0.80      0.88      0.84      1224
           2       0.92      0.93      0.93      1936

    accuracy                           0.87      3468
   macro avg       0.87      0.76      0.80      3468
weighted avg       0.88      0.87      0.87      3468



**5.2: logistic regression**

In [42]:
logisticRegModel = LogisticRegression()
logisticRegModel.fit(X_train, y_train)
yPred = logisticRegModel.predict(X_test)
print("Logistic Regression Classification Report:\n", classification_report(y_test, yPred))

Logistic Regression Classification Report:
               precision    recall  f1-score   support

           0       0.73      0.33      0.45       308
           1       0.73      0.84      0.78      1224
           2       0.90      0.90      0.90      1936

    accuracy                           0.83      3468
   macro avg       0.79      0.69      0.71      3468
weighted avg       0.83      0.83      0.82      3468



**5.3: Naïve Bayes**

In [44]:
naiveBayesModel = ComplementNB()

naiveBayesModel.fit(X_train, y_train)
yPred = naiveBayesModel.predict(X_test)
print("Naive Bayes Classification Report:\n", classification_report(y_test, yPred))

Naive Bayes Classification Report:
               precision    recall  f1-score   support

           0       0.45      0.28      0.34       308
           1       0.66      0.59      0.62      1224
           2       0.79      0.89      0.84      1936

    accuracy                           0.73      3468
   macro avg       0.63      0.59      0.60      3468
weighted avg       0.71      0.73      0.72      3468



In [25]:

def newReviewLabel(vectorizer, model, newReview):
    newReview = preprocess_text(newReview) #preprocess the new review

    vect_review = vectorizer.transform([newReview]) # vectorize it using the fitted vectorizer

    reviewLabel = model.predict(vect_review)[0]

    categories = { 0: "negative", 1: "neutral", 2: "positive"}
    strLabel = categories[reviewLabel]

    return strLabel


# we used svm model because it is the highest accuracy
print(newReviewLabel(vectorizer,svmModel,"excellent for price"))
print(newReviewLabel(vectorizer,svmModel,"easy to use"))
print(newReviewLabel(vectorizer,svmModel,"the center scroll wheel broke in weeks for no reason mouse was not dropped and barely used garbage so you get what you pay for"))
print(newReviewLabel(vectorizer,svmModel,"i love it and my friends use it from me"))
print(newReviewLabel(vectorizer,svmModel,"it has a defects while using it but useful  "))
print(newReviewLabel(vectorizer,svmModel,"the product broked my back instead of recover it's very bad"))




positive
neutral
negative
positive
neutral
negative
